<a href="https://colab.research.google.com/github/heramb-sharma-19/Innovexis-Internship/blob/main/Bird_Species_Observation_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Name:- Bird Species Observation Analysis #
##### **Project Type** - Data Cleaning and Preprocessing, Exploratory Data Analysis (EDA), Data Visualization, Geographic Analysis ,Species Analysis, SQL, Streamlit or PowerBI
##### **Contribution** - Individual
##### **Submitted By** - Heramb Sharma

## Git Hub:- ##
https://github.com/heramb-sharma-19/Innovexis-Internship/commits?author=heramb-sharma-19


## Problem Statement: ##
The project aims to analyze the distribution and diversity of bird species in two distinct ecosystems: forests and grasslands. By examining bird species observations across these habitats, the goal is to understand how environmental factors, such as vegetation type, climate, and terrain, influence bird populations and their behavior. The study will involve working on the provided observational data of bird species present in both ecosystems, identifying patterns of habitat preference, and assessing the impact of these habitats on bird diversity. The findings can provide valuable insights into habitat conservation, biodiversity management, and the effects of environmental changes on avian communities.


# Summary:- #
The Bird Species Observation Analysis project is a comprehensive data engineering and analytical ecosystem designed to monitor and evaluate avian populations across diverse natural environments. Centered around forest and grassland habitats, this project bridges the gap between raw, disconnected ecological field surveys and interactive, modern data products. By uniting data cleaning, SQL database design, scientific analysis, and interactive visualization, it provides conservationists and park administrators with a powerful lens to study avian activity, environmental dynamics, and urgent conservation needs.

The journey of the project begins with raw datasets stored in multi-sheet Excel files: [Bird_Monitoring_Data_FOREST.XLSX](file:///c:/Users/hp/OneDrive/Desktop/Zomato%20Project/p3/Bird_Monitoring_Data_FOREST.XLSX) and [Bird_Monitoring_Data_GRASSLAND.XLSX](file:///c:/Users/hp/OneDrive/Desktop/Zomato%20Project/p3/Bird_Monitoring_Data_GRASSLAND.XLSX). Within the main analytical notebook, [Bird_Species_Observation_Analysis.ipynb](file:///c:/Users/hp/OneDrive/Desktop/Zomato%20Project/p3/Bird_Species_Observation_Analysis.ipynb), these disparate sources are systematically ingested and merged. The pipeline cleans the combined records by resolving duplicate entries, standardizing taxonomic codes, and addressing missing values for critical observation attributes like distance, bird sex, and identification methods. Additionally, the data is geo-enriched by mapping administrative park codes—such as Rock Creek Park and Antietam National Battlefield—to their full names and geographic coordinates. Temporal features are engineered to track seasonal patterns, diurnal blocks, and survey durations. The resulting high-quality dataset is saved as [cleaned_bird_monitoring_data.csv](file:///c:/Users/hp/OneDrive/Desktop/Zomato%20Project/p3/cleaned_bird_monitoring_data.csv) and hosted within an indexed SQLite database, [bird_observations.db](file:///c:/Users/hp/OneDrive/Desktop/Zomato%20Project/p3/bird_observations.db), ensuring lightning-fast search capabilities.

The core of the analysis reveals how environmental conditions, timing, and human presence influence bird activity. SQL queries explore the data to trace daily and seasonal cycles, revealing when and where species are most active. This exploratory phase maps species distribution, analyzes detection ratios, and correlates sighting frequency with temperature, humidity, and environmental disturbances. Crucially, the project isolates priority conservation species, identifying birds flagged by the Partners in Flight Watchlist and Regional Stewardship initiatives.

To make these insights accessible, the project features an interactive web application in [app.py](file:///c:/Users/hp/OneDrive/Desktop/Zomato%20Project/p3/app.py). Built with Streamlit and styled with a premium dark theme, the dashboard offers a digital observatory of the parks. Users filter observations by ecosystem type, location, year, or species, while real-time metrics highlight total sightings and unique species. Key insights are presented across six structured pages: a Mapbox geolocational hotspot tracker, a temporal analysis of seasonal and hourly trends, a spatial plot analysis of forest versus grassland habitats, a behavioral dashboard showcasing sex ratios and flyover rates, an environmental section correlating weather variables, and a conservation center dedicated to tracking vulnerable species. Ultimately, the project turns complex wildlife datasets into actionable, beautiful tools to support biodiversity preservation.

## Business- Use Cases
1. Wildlife Conservation: Inform decisions on protecting critical bird habitats and enhancing biodiversity conservation efforts.
2. Land Management: Optimize land use and habitat restoration strategies by understanding the preferences of different bird species.
3. Eco-Tourism: Identify bird-rich areas to develop bird-watching tourism, attracting eco-tourists and boosting local economies.
4. Sustainable Agriculture: Support the development of agricultural practices that minimize the impact on bird populations in grasslands and forests.
5. Policy Support: Provide data-driven insights to help environmental agencies create effective conservation policies and strategies for vulnerable bird species.
6. Biodiversity Monitoring: Track the health and diversity of avian populations, aiding in the monitoring of ecosystem stability.


## Setup and Prerequisites

In [ ]:
# Install dependencies
!pip install -q pandas openpyxl plotly streamlit sqlalchemy

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px
import plotly.graph_objects as go
from datetime import time
import os

## 1. Data Loading and Merging
The dataset spans multiple sheets in two separate Excel workbooks. We will write a function to load all sheets from each workbook and combine them.

In [ ]:
forest_path = "Bird_Monitoring_Data_FOREST.XLSX"
grassland_path = "Bird_Monitoring_Data_GRASSLAND.XLSX"

def load_and_combine(file_path, default_location_type):
    xls = pd.ExcelFile(file_path)
    dfs = []
    for sheet in xls.sheet_names:
        df = xls.parse(sheet)
        if not df.empty:
            dfs.append(df)
    combined = pd.concat(dfs, ignore_index=True)
    combined['Location_Type'] = default_location_type
    return combined

if not os.path.exists(forest_path) or not os.path.exists(grassland_path):
    print("WARNING: Make sure you upload 'Bird_Monitoring_Data_FOREST.XLSX' and 'Bird_Monitoring_Data_GRASSLAND.XLSX' to your current folder!")
else:
    print("Files detected successfully.")

## 2. Data Cleaning & Preprocessing
We will clean the merged dataset by:
- Dropping duplicate rows.
- Harmonizing field names (e.g., merging `NPSTaxonCode` and `TaxonCode`).
- Handling missing values for Sub-Unit Code, Site Name, Sex, Distance, AcceptedTSN, TaxonCode, and ID Method.
- Engineering new temporal columns: `Month`, `Day`, `Season`, `Hour`, `Time_of_Day`, and `Duration_Minutes`.
- Converting all column headers to `snake_case` to ensure easy SQL querying.

In [ ]:
if os.path.exists(forest_path) and os.path.exists(grassland_path):
    forest_df = load_and_combine(forest_path, 'Forest')
    grassland_df = load_and_combine(grassland_path, 'Grassland')

    # Standardize columns
    forest_df = forest_df.rename(columns={'NPSTaxonCode': 'TaxonCode'})
    forest_df['Previously_Obs'] = False
    grassland_df['Site_Name'] = 'Grassland Site'

    # Combine datasets
    combined_df = pd.concat([forest_df, grassland_df], ignore_index=True)

    # Drop duplicates
    print(f"Shape before dropping duplicates: {combined_df.shape}")
    combined_df = combined_df.drop_duplicates()
    print(f"Shape after dropping duplicates: {combined_df.shape}")

    # Handle missing values
    combined_df['Sub_Unit_Code'] = combined_df['Sub_Unit_Code'].fillna('N/A')
    combined_df['Site_Name'] = combined_df['Site_Name'].fillna('Unknown')
    combined_df['Sex'] = combined_df['Sex'].fillna('Undetermined').replace({'U': 'Undetermined', 'unknown': 'Undetermined', 'Unknown': 'Undetermined'})
    combined_df['Distance'] = combined_df['Distance'].fillna('Unknown')
    combined_df['AcceptedTSN'] = combined_df['AcceptedTSN'].fillna(-1).astype(int)
    combined_df['TaxonCode'] = combined_df['TaxonCode'].fillna(-1).astype(int)
    combined_df['ID_Method'] = combined_df['ID_Method'].fillna('Unknown')
    combined_df['Previously_Obs'] = combined_df['Previously_Obs'].fillna(False).astype(bool)

    # Cast types
    bool_cols = ['Flyover_Observed', 'PIF_Watchlist_Status', 'Regional_Stewardship_Status', 'Initial_Three_Min_Cnt']
    for col in bool_cols:
        combined_df[col] = combined_df[col].fillna(False).astype(bool)

    combined_df['Year'] = pd.to_numeric(combined_df['Year'], errors='coerce').fillna(2018).astype(int)
    combined_df['Visit'] = pd.to_numeric(combined_df['Visit'], errors='coerce').fillna(1).astype(int)

    # Feature Engineering
    combined_df['Month'] = combined_df['Date'].dt.month.astype(int)
    combined_df['Day'] = combined_df['Date'].dt.day.astype(int)

    def get_season(month):
        if month in [12, 1, 2]: return 'Winter'
        elif month in [3, 4, 5]: return 'Spring'
        elif month in [6, 7, 8]: return 'Summer'
        else: return 'Autumn'
    combined_df['Season'] = combined_df['Month'].apply(get_season)

    combined_df['Hour'] = combined_df['Start_Time'].apply(lambda x: x.hour if isinstance(x, time) else pd.to_datetime(x).hour).astype(int)

    def get_time_of_day(hour):
        if 5 <= hour <= 8: return 'Early Morning'
        elif 9 <= hour <= 11: return 'Late Morning'
        elif 12 <= hour <= 16: return 'Afternoon'
        elif 17 <= hour <= 20: return 'Evening'
        else: return 'Night'
    combined_df['Time_of_Day'] = combined_df['Hour'].apply(get_time_of_day)

    def calc_duration(row):
        try:
            t1, t2 = row['Start_Time'], row['End_Time']
            m1 = t1.hour * 60 + t1.minute
            m2 = t2.hour * 60 + t2.minute
            diff = m2 - m1
            return diff + 1440 if diff < 0 else diff
        except:
            return 0
    combined_df['Duration_Minutes'] = combined_df.apply(calc_duration, axis=1).astype(int)

    # Add geocoding and name mapping for administrative units
    admin_unit_info = {
        'ANTI': {'name': 'Antietam National Battlefield', 'lat': 39.4678, 'lon': -77.7422},
        'CATO': {'name': 'Catoctin Mountain Park', 'lat': 39.6362, 'lon': -77.4528},
        'CHOH': {'name': 'Chesapeake and Ohio Canal National Historical Park', 'lat': 39.5447, 'lon': -77.8384},
        'GWMP': {'name': 'George Washington Memorial Parkway', 'lat': 38.9281, 'lon': -77.0872},
        'HAFE': {'name': 'Harpers Ferry National Historical Park', 'lat': 39.3242, 'lon': -77.7303},
        'MANA': {'name': 'Manassas National Battlefield Park', 'lat': 38.8130, 'lon': -77.5146},
        'MONO': {'name': 'Monocacy National Battlefield', 'lat': 39.3734, 'lon': -77.3941},
        'NACE': {'name': 'National Capital East Parks', 'lat': 38.8683, 'lon': -76.9631},
        'PRWI': {'name': 'Prince William Forest Park', 'lat': 38.5668, 'lon': -77.3820},
        'ROCR': {'name': 'Rock Creek Park', 'lat': 38.9667, 'lon': -77.0425},
        'WOTR': {'name': 'Wolf Trap National Park for the Performing Arts', 'lat': 38.9387, 'lon': -77.2654}
    }
    combined_df['Admin_Unit_Name'] = combined_df['Admin_Unit_Code'].map(lambda x: admin_unit_info.get(x, {}).get('name', 'Unknown'))
    combined_df['Latitude'] = combined_df['Admin_Unit_Code'].map(lambda x: admin_unit_info.get(x, {}).get('lat', np.nan))
    combined_df['Longitude'] = combined_df['Admin_Unit_Code'].map(lambda x: admin_unit_info.get(x, {}).get('lon', np.nan))

    # Rename to snake_case
    rename_dict = {
        'Admin_Unit_Code': 'admin_unit_code', 'Sub_Unit_Code': 'sub_unit_code',
        'Site_Name': 'site_name', 'Plot_Name': 'plot_name',
        'Location_Type': 'location_type', 'Year': 'year', 'Date': 'date',
        'Start_Time': 'start_time', 'End_Time': 'end_time', 'Observer': 'observer',
        'Visit': 'visit', 'Interval_Length': 'interval_length', 'ID_Method': 'id_method',
        'Distance': 'distance', 'Flyover_Observed': 'flyover_observed', 'Sex': 'sex',
        'Common_Name': 'common_name', 'Scientific_Name': 'scientific_name',
        'AcceptedTSN': 'accepted_tsn', 'TaxonCode': 'taxon_code', 'AOU_Code': 'aou_code',
        'PIF_Watchlist_Status': 'pif_watchlist_status',
        'Regional_Stewardship_Status': 'regional_stewardship_status',
        'Temperature': 'temperature', 'Humidity': 'humidity', 'Sky': 'sky',
        'Wind': 'wind', 'Disturbance': 'disturbance', 'Initial_Three_Min_Cnt': 'initial_three_min_cnt',
        'Previously_Obs': 'previously_obs', 'Admin_Unit_Name': 'admin_unit_name',
        'Latitude': 'latitude', 'Longitude': 'longitude'
    }
    cleaned_df = combined_df.rename(columns=rename_dict)
    cleaned_df.to_csv('cleaned_bird_monitoring_data.csv', index=False)
    print("Cleaned dataset saved as CSV. Shape:", cleaned_df.shape)
else:
    # Load from the local pre-generated CSV if in local run
    cleaned_df = pd.read_csv('cleaned_bird_monitoring_data.csv')
    print("Loaded pre-cleaned CSV data. Shape:", cleaned_df.shape)

## 3. Database Storage
We store our cleaned data in an SQLite database `bird_observations.db` and set up performance indexes.

In [ ]:
db_path = "bird_observations.db"
conn = sqlite3.connect(db_path)

df_to_sql = cleaned_df.copy()
if 'date' in df_to_sql.columns and pd.api.types.is_datetime64_any_dtype(df_to_sql['date']):
    df_to_sql['date'] = df_to_sql['date'].dt.strftime('%Y-%m-%d')

df_to_sql['start_time'] = df_to_sql['start_time'].apply(lambda x: x.strftime('%H:%M:%S') if isinstance(x, time) else str(x))
df_to_sql['end_time'] = df_to_sql['end_time'].apply(lambda x: x.strftime('%H:%M:%S') if isinstance(x, time) else str(x))

df_to_sql.to_sql('bird_observations', conn, if_exists='replace', index=False)

# Create indexes for optimized querying
cursor = conn.cursor()
cursor.execute("CREATE INDEX IF NOT EXISTS idx_species ON bird_observations (common_name);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_location_type ON bird_observations (location_type);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_year ON bird_observations (year);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_admin_unit ON bird_observations (admin_unit_code);")
conn.commit()
print("Data loaded to SQLite table 'bird_observations' with indexes.")

## 4. SQL-Based Exploratory Data Analysis (EDA)
We run SQL queries to gain key ecological insights and visualize the results using Plotly.

### 4.1 Temporal Analysis
Analyzing seasonal and hourly sighting trends.

In [ ]:
print("--- Sighting counts by Season ---")
query_season = """
SELECT season, COUNT(*) as sighting_count, COUNT(DISTINCT common_name) as unique_species
FROM bird_observations
GROUP BY season
ORDER BY sighting_count DESC
"""
df_season = pd.read_sql_query(query_season, conn)
print(df_season)

fig_season = px.bar(df_season, x='season', y='sighting_count', color='season',
                     title='Bird Sightings by Season', text_auto=True,
                     labels={'season': 'Season', 'sighting_count': 'Total Sightings'},
                     color_discrete_sequence=px.colors.qualitative.Dark24)
fig_season.show()

print("\n--- Sighting counts by Hour ---")
query_hour = """
SELECT hour, COUNT(*) as sighting_count
FROM bird_observations
GROUP BY hour
ORDER BY hour
"""
df_hour = pd.read_sql_query(query_hour, conn)
fig_hour = px.line(df_hour, x='hour', y='sighting_count', markers=True,
                    title='Hourly Sighting Frequency',
                    labels={'hour': 'Hour of Day (24h)', 'sighting_count': 'Total Sightings'})
fig_hour.update_traces(line_color='#2ca02c')
fig_hour.show()

### 4.2 Spatial Analysis
Evaluating species richness and hotspots across ecosystems and plots.

In [ ]:
print("--- Sighting counts by Ecosystem Type ---")
query_eco = """
SELECT location_type, COUNT(*) as sighting_count, COUNT(DISTINCT common_name) as unique_species
FROM bird_observations
GROUP BY location_type
"""
df_eco = pd.read_sql_query(query_eco, conn)
print(df_eco)

fig_eco = px.pie(df_eco, values='sighting_count', names='location_type',
                 title='Distribution of Sightings by Ecosystem',
                 hole=0.4, color_discrete_map={'Forest': '#10B981', 'Grassland': '#F59E0B'})
fig_eco.show()

print("\n--- Sighting counts by Admin Unit ---")
query_admin = """
SELECT admin_unit_code, COUNT(*) as sighting_count, COUNT(DISTINCT common_name) as unique_species
FROM bird_observations
GROUP BY admin_unit_code
ORDER BY sighting_count DESC
"""
df_admin = pd.read_sql_query(query_admin, conn)
print(df_admin)

fig_admin = px.bar(df_admin, x='admin_unit_code', y='sighting_count', color='unique_species',
                   title='Total Sightings and Species Richness by Administrative Unit',
                   color_continuous_scale='Viridis',
                   labels={'admin_unit_code': 'Admin Unit Code', 'sighting_count': 'Total Sightings', 'unique_species': 'Unique Species'})
fig_admin.show()

print("\n--- Top 10 Sighting Hotspots (Plots) ---")
query_plots = """
SELECT plot_name, location_type, COUNT(*) as sighting_count
FROM bird_observations
GROUP BY plot_name, location_type
ORDER BY sighting_count DESC
LIMIT 10
"""
df_plots = pd.read_sql_query(query_plots, conn)
print(df_plots)

fig_plots = px.bar(df_plots, x='plot_name', y='sighting_count', color='location_type',
                   title='Top 10 Sighting Hotspots (Plots)',
                   labels={'plot_name': 'Plot Name', 'sighting_count': 'Total Sightings'},
                   color_discrete_map={'Forest': '#10B981', 'Grassland': '#F59E0B'})
fig_plots.show()

### 4.3 Species Analysis
Let's look at the most common species, identification methods, and sex distribution.

In [ ]:
print("--- Top 10 Most Common Bird Species ---")
query_common = """
SELECT common_name, scientific_name, COUNT(*) as sighting_count
FROM bird_observations
GROUP BY common_name, scientific_name
ORDER BY sighting_count DESC
LIMIT 10
"""
df_common = pd.read_sql_query(query_common, conn)
print(df_common)

fig_common = px.bar(df_common, y='common_name', x='sighting_count', orientation='h',
                    title='Top 10 Most Common Bird Species',
                    labels={'common_name': 'Common Name', 'sighting_count': 'Total Sightings'},
                    color='sighting_count', color_continuous_scale='Cividis')
fig_common.update_layout(yaxis={'categoryorder':'total ascending'})
fig_common.show()

print("\n--- Identification Methods ---")
query_id = """
SELECT id_method, COUNT(*) as count
FROM bird_observations
GROUP BY id_method
ORDER BY count DESC
"""
df_id = pd.read_sql_query(query_id, conn)
print(df_id)

print("\n--- Sex Distribution for Top 5 Species ---")
query_sex = """
SELECT common_name, sex, COUNT(*) as count
FROM bird_observations
WHERE common_name IN (
    SELECT common_name FROM bird_observations
    GROUP BY common_name ORDER BY COUNT(*) DESC LIMIT 5
) AND sex IN ('Male', 'Female', 'Undetermined')
GROUP BY common_name, sex
"""
df_sex = pd.read_sql_query(query_sex, conn)
fig_sex = px.bar(df_sex, x='common_name', y='count', color='sex', barmode='group',
                 title='Sex Distribution for Top 5 Most Common Species',
                 labels={'common_name': 'Species', 'count': 'Count', 'sex': 'Sex'})
fig_sex.show()

### 4.4 Environmental Conditions
Examining how temperature, humidity, sky cover, wind, and disturbances influence bird observations.

In [ ]:
print("--- Sighting counts by Sky Condition ---")
query_sky = """
SELECT sky, COUNT(*) as sighting_count, AVG(temperature) as avg_temp, AVG(humidity) as avg_humidity
FROM bird_observations
GROUP BY sky
ORDER BY sighting_count DESC
"""
df_sky = pd.read_sql_query(query_sky, conn)
print(df_sky)

print("\n--- Sighting Count by Disturbance ---")
query_dist = """
SELECT disturbance, COUNT(*) as sighting_count, COUNT(DISTINCT common_name) as unique_species
FROM bird_observations
GROUP BY disturbance
ORDER BY sighting_count DESC
"""
df_dist = pd.read_sql_query(query_dist, conn)
print(df_dist)

print("\n--- Sighting Density by Temperature & Humidity ---")
query_temp_hum = """
SELECT temperature, humidity, COUNT(*) as sighting_count
FROM bird_observations
GROUP BY temperature, humidity
"""
df_temp_hum = pd.read_sql_query(query_temp_hum, conn)
fig_temp_hum = px.scatter(df_temp_hum, x='temperature', y='humidity', size='sighting_count', color='sighting_count',
                          title='Sighting Density by Temperature and Humidity',
                          labels={'temperature': 'Temperature (°F)', 'humidity': 'Humidity (%)'},
                          color_continuous_scale='Viridis')
fig_temp_hum.show()

### 4.5 Distance and Behavior
Evaluating the distance classes of sightings and flyover rates by ecosystem.

In [ ]:
print("--- Sighting Distance by Ecosystem ---")
query_dist_loc = """
SELECT distance, location_type, COUNT(*) as count
FROM bird_observations
GROUP BY distance, location_type
ORDER BY count DESC
"""
df_dist_loc = pd.read_sql_query(query_dist_loc, conn)
print(df_dist_loc)

print("\n--- Flyover Observations by Ecosystem ---")
query_fly = """
SELECT flyover_observed, location_type, COUNT(*) as count
FROM bird_observations
GROUP BY flyover_observed, location_type
"""
df_fly = pd.read_sql_query(query_fly, conn)
print(df_fly)

### 4.6 Observer Trends & Visits
Analyzing observer sightings and the impact of repeated visits on species richness.

In [ ]:
print("--- Top Observers by Sighting Count ---")
query_obs = """
SELECT observer, COUNT(*) as sighting_count, COUNT(DISTINCT common_name) as unique_species
FROM bird_observations
GROUP BY observer
ORDER BY sighting_count DESC
LIMIT 10
"""
df_obs = pd.read_sql_query(query_obs, conn)
print(df_obs)

print("\n--- Sightings and Avg Species Richness by Visit ---")
query_visit = """
SELECT visit, COUNT(*) as sighting_count, AVG(species_per_visit) as avg_unique_species
FROM (
    SELECT visit, plot_name, year, COUNT(DISTINCT common_name) as species_per_visit
    FROM bird_observations
    GROUP BY visit, plot_name, year
)
GROUP BY visit
ORDER BY visit
"""
df_visit = pd.read_sql_query(query_visit, conn)
print(df_visit)

### 4.7 Conservation Insights
Filtering and analyzing species on the Partners in Flight (PIF) Watchlist or having Regional Stewardship priority.

In [ ]:
print("--- Top At-Risk / Regional Stewardship Species by Sightings ---")
query_conserve = """
SELECT common_name, scientific_name, location_type, pif_watchlist_status, regional_stewardship_status, COUNT(*) as sighting_count
FROM bird_observations
WHERE pif_watchlist_status = 1 OR regional_stewardship_status = 1
GROUP BY common_name, scientific_name, location_type
ORDER BY sighting_count DESC
LIMIT 15
"""
df_conserve = pd.read_sql_query(query_conserve, conn)
print(df_conserve)

fig_conserve = px.bar(df_conserve, x='common_name', y='sighting_count', color='location_type',
                      title='Top 15 At-Risk or Regional Stewardship Species by Sightings',
                      labels={'common_name': 'Species Common Name', 'sighting_count': 'Sighting Count'},
                      color_discrete_map={'Forest': '#10B981', 'Grassland': '#F59E0B'})
fig_conserve.show()

## 5. Exporting the Streamlit App Code
Run the cell below to write the `app.py` script to the local environment so you can run the Streamlit dashboard.

In [ ]:
streamlit_code = 'import streamlit as st\nimport pandas as pd\nimport sqlite3\nimport plotly.express as px\nimport plotly.graph_objects as go\n\n# Set Page Config\nst.set_page_config(\n    page_title="Bird Species Observation Analysis",\n    page_icon="🐦",\n    layout="wide",\n    initial_sidebar_state="expanded"\n)\n\n# Custom CSS for dark-themed, premium UI\nst.markdown("""\n<style>\n    /* Main container background and text color */\n    .stApp {\n        background-color: #0e1117;\n        color: #ecf0f1;\n    }\n    \n    /* Sidebar styling */\n    [data-testid="stSidebar"] {\n        background-color: #161b22;\n        border-right: 1px solid #30363d;\n    }\n    \n    /* Title and header styling */\n    h1, h2, h3, h4, h5, h6 {\n        color: #58a6ff !important;\n        font-family: \'Inter\', sans-serif;\n    }\n    \n    /* Custom metric card */\n    .metric-card {\n        background-color: #1f242d;\n        border: 1px solid #30363d;\n        border-radius: 10px;\n        padding: 20px;\n        text-align: center;\n        box-shadow: 0 4px 6px rgba(0,0,0,0.3);\n        transition: transform 0.3s ease, border-color 0.3s ease;\n    }\n    .metric-card:hover {\n        transform: translateY(-4px);\n        border-color: #58a6ff;\n    }\n    .metric-title {\n        color: #8b949e;\n        font-size: 14px;\n        font-weight: 600;\n        text-transform: uppercase;\n        margin-bottom: 8px;\n    }\n    .metric-value {\n        color: #58a6ff;\n        font-size: 28px;\n        font-weight: 700;\n    }\n    \n    /* Tabs styling */\n    .stTabs [data-baseweb="tab-list"] {\n        gap: 8px;\n    }\n    .stTabs [data-baseweb="tab"] {\n        background-color: #1f242d;\n        border: 1px solid #30363d;\n        border-radius: 6px 6px 0px 0px;\n        color: #8b949e;\n        padding: 10px 20px;\n        font-weight: 600;\n    }\n    .stTabs [aria-selected="true"] {\n        background-color: #58a6ff !important;\n        color: #0e1117 !important;\n        border-color: #58a6ff !important;\n    }\n</style>\n""", unsafe_value_html=True)\n\n# Helper function to create DB connection\ndef get_connection():\n    return sqlite3.connect("bird_observations.db")\n\n# Load cached initial configuration data\n@st.cache_data\ndef get_filter_options():\n    conn = get_connection()\n    df_filters = pd.read_sql_query(\n        "SELECT DISTINCT location_type, admin_unit_code, year, common_name FROM bird_observations", \n        conn\n    )\n    conn.close()\n    \n    ecosystems = sorted(df_filters[\'location_type\'].unique())\n    admin_units = sorted(df_filters[\'admin_unit_code\'].unique())\n    years = sorted([int(y) for y in df_filters[\'year\'].unique()])\n    species = sorted(df_filters[\'common_name\'].unique())\n    return ecosystems, admin_units, years, species\n\n# Load filtered data based on sidebar settings\ndef load_filtered_data(selected_ecosystems, selected_admin_units, selected_years, selected_species, watchlist_filter, stewardship_filter):\n    conn = get_connection()\n    \n    query = "SELECT * FROM bird_observations WHERE 1=1"\n    params = []\n    \n    if selected_ecosystems:\n        query += f" AND location_type IN ({\',\'.join([\'?\']*len(selected_ecosystems))})"\n        params.extend(selected_ecosystems)\n        \n    if selected_admin_units:\n        query += f" AND admin_unit_code IN ({\',\'.join([\'?\']*len(selected_admin_units))})"\n        params.extend(selected_admin_units)\n        \n    if selected_years:\n        query += f" AND year IN ({\',\'.join([\'?\']*len(selected_years))})"\n        params.extend(selected_years)\n        \n    if selected_species != "All":\n        query += " AND common_name = ?"\n        params.append(selected_species)\n        \n    if watchlist_filter != "All":\n        val = 1 if watchlist_filter == "Yes" else 0\n        query += " AND pif_watchlist_status = ?"\n        params.append(val)\n        \n    if stewardship_filter != "All":\n        val = 1 if stewardship_filter == "Yes" else 0\n        query += " AND regional_stewardship_status = ?"\n        params.append(val)\n        \n    df = pd.read_sql_query(query, conn, params=params)\n    conn.close()\n    return df\n\n# Get filter data\necosystems, admin_units, years, species_list = get_filter_options()\n\n# --- SIDEBAR FILTERS ---\nst.sidebar.image("https://img.icons8.com/color/96/000000/bird.png", width=80)\nst.sidebar.title("Filter Options")\n\nselected_ecosystems = st.sidebar.multiselect("Ecosystem Type", options=ecosystems, default=ecosystems)\nselected_admin_units = st.sidebar.multiselect("Administrative Unit", options=admin_units, default=admin_units)\nselected_years = st.sidebar.multiselect("Observation Year", options=years, default=years)\n\nspecies_options = ["All"] + species_list\nselected_species = st.sidebar.selectbox("Specific Species", options=species_options)\n\nwatchlist_filter = st.sidebar.selectbox("PIF Watchlist Status", options=["All", "Yes", "No"])\nstewardship_filter = st.sidebar.selectbox("Regional Stewardship Status", options=["All", "Yes", "No"])\n\n# Load data\ndf = load_filtered_data(\n    selected_ecosystems, \n    selected_admin_units, \n    selected_years, \n    selected_species, \n    watchlist_filter, \n    stewardship_filter\n)\n\n# App Header\nst.title("🐦 Bird Species Observation Analysis")\nst.markdown("### Ecological Monitoring & Conservation Insights")\nst.markdown("Explore and analyze bird populations and their environmental drivers across forests and grasslands.")\n\nif df.empty:\n    st.warning("No data found matching the selected filters. Please adjust the sidebar options.")\nelse:\n    # --- METRICS SECTION ---\n    total_obs = len(df)\n    unique_species = df[\'common_name\'].nunique()\n    avg_temp = df[\'temperature\'].mean()\n    watchlist_count = df[\'pif_watchlist_status\'].sum()\n    \n    col1, col2, col3, col4 = st.columns(4)\n    with col1:\n        st.markdown(f\'<div class="metric-card"><div class="metric-title">Total Observations</div><div class="metric-value">{total_obs:,}</div></div>\', unsafe_value_html=True)\n    with col2:\n        st.markdown(f\'<div class="metric-card"><div class="metric-title">Unique Species</div><div class="metric-value">{unique_species:,}</div></div>\', unsafe_value_html=True)\n    with col3:\n        st.markdown(f\'<div class="metric-card"><div class="metric-title">Avg Temperature</div><div class="metric-value">{avg_temp:.1f}°F</div></div>\', unsafe_value_html=True)\n    with col4:\n        st.markdown(f\'<div class="metric-card"><div class="metric-title">Watchlist Sightings</div><div class="metric-value">{watchlist_count:,}</div></div>\', unsafe_value_html=True)\n        \n    st.markdown("<br>", unsafe_value_html=True)\n    \n    # --- TABS ---\n    tab1, tab2, tab3, tab4, tab5, tab6 = st.tabs([\n        "📍 Geographic & Overview", \n        "📅 Temporal Trends", \n        "🌳 Spatial & Habitat Analysis", \n        "🦜 Species & Behavior",\n        "🌦️ Weather & Disturbance",\n        "🛡️ Conservation Insights"\n    ])\n    \n    with tab1:\n        st.header("Geographic & Park Overview")\n        col_map, col_details = st.columns([2, 1])\n        \n        with col_map:\n            # Aggregate data by park\n            map_data = df.groupby([\'admin_unit_name\', \'latitude\', \'longitude\']).agg(\n                sighting_count=(\'common_name\', \'count\'),\n                unique_species=(\'common_name\', \'nunique\')\n            ).reset_index()\n            \n            fig_map = px.scatter_mapbox(\n                map_data, \n                lat="latitude", \n                lon="longitude", \n                size="sighting_count", \n                color="unique_species",\n                color_continuous_scale=px.colors.sequential.Viridis,\n                hover_name="admin_unit_name", \n                hover_data=["sighting_count", "unique_species"],\n                zoom=7.5, \n                height=500,\n                title="Sighting Hotspots & Richness by Park"\n            )\n            fig_map.update_layout(\n                mapbox_style="carto-darkmatter",\n                margin={"r":0,"t":40,"l":0,"b":0},\n                paper_bgcolor=\'rgba(0,0,0,0)\',\n                font_color="#ecf0f1"\n            )\n            st.plotly_chart(fig_map, use_container_width=True)\n            \n        with col_details:\n            st.subheader("Park Summary")\n            park_summary = df.groupby(\'admin_unit_code\').agg(\n                Observations=(\'common_name\', \'count\'),\n                Species=(\'common_name\', \'nunique\')\n            ).rename(columns={\'Observations\': \'Sightings\', \'Species\': \'Unique Species\'}).sort_values(by=\'Sightings\', ascending=False)\n            st.dataframe(park_summary, use_container_width=True)\n            \n    with tab2:\n        st.header("Temporal Activity Analysis")\n        col2_1, col2_2 = st.columns(2)\n        \n        with col2_1:\n            # Sighting count by Month\n            month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]\n            month_map = {1: \'Jan\', 2: \'Feb\', 3: \'Mar\', 4: \'Apr\', 5: \'May\', 6: \'Jun\', 7: \'Jul\', 8: \'Aug\', 9: \'Sep\', 10: \'Oct\', 11: \'Nov\', 12: \'Dec\'}\n            df[\'month_name\'] = df[\'month\'].map(month_map)\n            month_df = df.groupby(\'month_name\').size().reindex(month_order).fillna(0).reset_index(name=\'count\')\n            \n            fig_month = px.bar(\n                month_df, \n                x=\'month_name\', \n                y=\'count\', \n                title=\'Observations by Month\',\n                labels={\'month_name\': \'Month\', \'count\': \'Number of Observations\'},\n                color=\'count\',\n                color_continuous_scale=px.colors.sequential.Bluyl\n            )\n            fig_month.update_layout(paper_bgcolor=\'rgba(0,0,0,0)\', plot_bgcolor=\'rgba(0,0,0,0)\', font_color="#ecf0f1")\n            st.plotly_chart(fig_month, use_container_width=True)\n            \n        with col2_2:\n            # Sighting count by Hour\n            hour_df = df.groupby(\'hour\').size().reset_index(name=\'count\')\n            fig_hour = px.line(\n                hour_df, \n                x=\'hour\', \n                y=\'count\', \n                title=\'Sighting Frequency by Hour of Day\',\n                labels={\'hour\': \'Hour of Day (24h)\', \'count\': \'Number of Observations\'},\n                markers=True\n            )\n            fig_hour.update_traces(line_color=\'#58a6ff\', marker=dict(size=8, color=\'#38bdf8\'))\n            fig_hour.update_layout(paper_bgcolor=\'rgba(0,0,0,0)\', plot_bgcolor=\'rgba(0,0,0,0)\', font_color="#ecf0f1")\n            st.plotly_chart(fig_hour, use_container_width=True)\n            \n        # Seasonal Analysis\n        st.subheader("Seasonal Patterns")\n        season_df = df.groupby([\'season\', \'location_type\']).size().reset_index(name=\'count\')\n        fig_season = px.bar(\n            season_df, \n            x=\'season\', \n            y=\'count\', \n            color=\'location_type\',\n            barmode=\'group\',\n            title=\'Seasonal Sighting Distribution across Ecosystems\',\n            labels={\'season\': \'Season\', \'count\': \'Observations\', \'location_type\': \'Ecosystem\'},\n            color_discrete_map={\'Forest\': \'#10B981\', \'Grassland\': \'#F59E0B\'}\n        )\n        fig_season.update_layout(paper_bgcolor=\'rgba(0,0,0,0)\', plot_bgcolor=\'rgba(0,0,0,0)\', font_color="#ecf0f1")\n        st.plotly_chart(fig_season, use_container_width=True)\n        \n    with tab3:\n        st.header("Spatial & Plot Analysis")\n        col3_1, col3_2 = st.columns(2)\n        \n        with col3_1:\n            # Sighting counts by Ecosystem Type\n            eco_df = df.groupby(\'location_type\').size().reset_index(name=\'count\')\n            fig_eco = px.pie(\n                eco_df, \n                values=\'count\', \n                names=\'location_type\', \n                title=\'Ecosystem Sighting Distribution\',\n                hole=0.4,\n                color=\'location_type\',\n                color_discrete_map={\'Forest\': \'#10B981\', \'Grassland\': \'#F59E0B\'}\n            )\n            fig_eco.update_layout(paper_bgcolor=\'rgba(0,0,0,0)\', font_color="#ecf0f1")\n            st.plotly_chart(fig_eco, use_container_width=True)\n            \n        with col3_2:\n            # Top Plots (Hotspots)\n            top_plots = df.groupby([\'plot_name\', \'location_type\']).size().reset_index(name=\'count\').sort_values(by=\'count\', ascending=False).head(15)\n            fig_plots = px.bar(\n                top_plots, \n                x=\'count\', \n                y=\'plot_name\', \n                color=\'location_type\', \n                orientation=\'h\',\n                title=\'Top 15 Plot Hotspots\',\n                labels={\'plot_name\': \'Plot Name\', \'count\': \'Observations\', \'location_type\': \'Ecosystem\'},\n                color_discrete_map={\'Forest\': \'#10B981\', \'Grassland\': \'#F59E0B\'}\n            )\n            fig_plots.update_layout(yaxis={\'categoryorder\':\'total ascending\'}, paper_bgcolor=\'rgba(0,0,0,0)\', plot_bgcolor=\'rgba(0,0,0,0)\', font_color="#ecf0f1")\n            st.plotly_chart(fig_plots, use_container_width=True)\n            \n    with tab4:\n        st.header("Species & Behavioral Insights")\n        col4_1, col4_2 = st.columns(2)\n        \n        with col4_1:\n            # Top 15 most common species\n            top_species = df.groupby([\'common_name\', \'scientific_name\']).size().reset_index(name=\'count\').sort_values(by=\'count\', ascending=False).head(15)\n            fig_species = px.bar(\n                top_species, \n                x=\'count\', \n                y=\'common_name\', \n                orientation=\'h\',\n                title=\'Top 15 Most Common Bird Species\',\n                labels={\'common_name\': \'Common Name\', \'count\': \'Total Observations\'},\n                color=\'count\',\n                color_continuous_scale=px.colors.sequential.Viridis\n            )\n            fig_species.update_layout(yaxis={\'categoryorder\':\'total ascending\'}, paper_bgcolor=\'rgba(0,0,0,0)\', plot_bgcolor=\'rgba(0,0,0,0)\', font_color="#ecf0f1")\n            st.plotly_chart(fig_species, use_container_width=True)\n            \n        with col4_2:\n            # ID Methods\n            id_df = df.groupby(\'id_method\').size().reset_index(name=\'count\')\n            fig_id = px.pie(\n                id_df, \n                values=\'count\', \n                names=\'id_method\', \n                title=\'Distribution of Identification Methods\',\n                hole=0.4,\n                color_discrete_sequence=px.colors.qualitative.Safe\n            )\n            fig_id.update_layout(paper_bgcolor=\'rgba(0,0,0,0)\', font_color="#ecf0f1")\n            st.plotly_chart(fig_id, use_container_width=True)\n            \n        col4_3, col4_4 = st.columns(2)\n        with col4_3:\n            # Sex Ratio\n            sex_df = df.groupby([\'sex\', \'location_type\']).size().reset_index(name=\'count\')\n            fig_sex = px.bar(\n                sex_df, \n                x=\'sex\', \n                y=\'count\', \n                color=\'location_type\', \n                barmode=\'group\',\n                title=\'Sex Distribution by Ecosystem\',\n                labels={\'sex\': \'Sex Classification\', \'count\': \'Count\', \'location_type\': \'Ecosystem\'},\n                color_discrete_map={\'Forest\': \'#10B981\', \'Grassland\': \'#F59E0B\'}\n            )\n            fig_sex.update_layout(paper_bgcolor=\'rgba(0,0,0,0)\', plot_bgcolor=\'rgba(0,0,0,0)\', font_color="#ecf0f1")\n            st.plotly_chart(fig_sex, use_container_width=True)\n            \n        with col4_4:\n            # Flyover Observed\n            fly_df = df.groupby([\'flyover_observed\', \'location_type\']).size().reset_index(name=\'count\')\n            fig_fly = px.bar(\n                fly_df, \n                x=\'flyover_observed\', \n                y=\'count\', \n                color=\'location_type\', \n                barmode=\'group\',\n                title=\'Flyover Observations by Ecosystem\',\n                labels={\'flyover_observed\': \'Flyover Observed (True/False)\', \'count\': \'Sighting Count\', \'location_type\': \'Ecosystem\'},\n                color_discrete_map={\'Forest\': \'#10B981\', \'Grassland\': \'#F59E0B\'}\n            )\n            fig_fly.update_layout(paper_bgcolor=\'rgba(0,0,0,0)\', plot_bgcolor=\'rgba(0,0,0,0)\', font_color="#ecf0f1")\n            st.plotly_chart(fig_fly, use_container_width=True)\n            \n    with tab5:\n        st.header("Weather & Environmental Correlations")\n        col5_1, col5_2 = st.columns(2)\n        \n        with col5_1:\n            # Scatter Plot: Temp vs Humidity colored by Ecosystem\n            fig_temp_hum = px.scatter(\n                df, \n                x=\'temperature\', \n                y=\'humidity\', \n                color=\'location_type\', \n                opacity=0.5,\n                title=\'Temperature vs Humidity Sighting Distribution\',\n                labels={\'temperature\': \'Temperature (°F)\', \'humidity\': \'Humidity (%)\', \'location_type\': \'Ecosystem\'},\n                color_discrete_map={\'Forest\': \'#10B981\', \'Grassland\': \'#F59E0B\'}\n            )\n            fig_temp_hum.update_layout(paper_bgcolor=\'rgba(0,0,0,0)\', plot_bgcolor=\'rgba(0,0,0,0)\', font_color="#ecf0f1")\n            st.plotly_chart(fig_temp_hum, use_container_width=True)\n            \n        with col5_2:\n            # Disturbance impact on sightings\n            dist_df = df.groupby([\'disturbance\', \'location_type\']).size().reset_index(name=\'count\')\n            fig_dist = px.bar(\n                dist_df, \n                x=\'disturbance\', \n                y=\'count\', \n                color=\'location_type\', \n                barmode=\'group\',\n                title=\'Bird Observations under Different Disturbance Levels\',\n                labels={\'disturbance\': \'Disturbance Level\', \'count\': \'Total Observations\', \'location_type\': \'Ecosystem\'},\n                color_discrete_map={\'Forest\': \'#10B981\', \'Grassland\': \'#F59E0B\'}\n            )\n            fig_dist.update_layout(paper_bgcolor=\'rgba(0,0,0,0)\', plot_bgcolor=\'rgba(0,0,0,0)\', font_color="#ecf0f1")\n            st.plotly_chart(fig_dist, use_container_width=True)\n\n    with tab6:\n        st.header("Conservation & Watchlist Status")\n        \n        col6_1, col6_2 = st.columns(2)\n        with col6_1:\n            st.subheader("Partners in Flight (PIF) Watchlist Species")\n            pif_df = df[df[\'pif_watchlist_status\'] == 1].groupby([\'common_name\', \'scientific_name\']).size().reset_index(name=\'sightings\').sort_values(by=\'sightings\', ascending=False)\n            st.dataframe(pif_df, use_container_width=True)\n            \n        with col6_2:\n            st.subheader("Regional Stewardship Priority Species")\n            steward_df = df[df[\'regional_stewardship_status\'] == 1].groupby([\'common_name\', \'scientific_name\']).size().reset_index(name=\'sightings\').sort_values(by=\'sightings\', ascending=False)\n            st.dataframe(steward_df, use_container_width=True)\n            \n        st.markdown("---")\n        # Visualizing proportion of at-risk sightings\n        conserve_summary = pd.DataFrame({\n            \'Category\': [\'Normal Species\', \'PIF Watchlist Only\', \'Regional Stewardship Only\', \'Both Watchlist & Stewardship\'],\n            \'Count\': [\n                len(df[(df[\'pif_watchlist_status\'] == 0) & (df[\'regional_stewardship_status\'] == 0)]),\n                len(df[(df[\'pif_watchlist_status\'] == 1) & (df[\'regional_stewardship_status\'] == 0)]),\n                len(df[(df[\'pif_watchlist_status\'] == 0) & (df[\'regional_stewardship_status\'] == 1)]),\n                len(df[(df[\'pif_watchlist_status\'] == 1) & (df[\'regional_stewardship_status\'] == 1)])\n            ]\n        })\n        fig_conserve = px.pie(\n            conserve_summary, \n            values=\'Count\', \n            names=\'Category\', \n            title=\'Proportion of At-Risk and Priority Sighting Categories\',\n            hole=0.4,\n            color_discrete_sequence=px.colors.qualitative.Dark24\n        )\n        fig_conserve.update_layout(paper_bgcolor=\'rgba(0,0,0,0)\', font_color="#ecf0f1")\n        st.plotly_chart(fig_conserve, use_container_width=True)\n'
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_code)
print('app.py successfully written!')

## 6. How to Run the Streamlit Dashboard in Google Colab

To launch and view your Streamlit application directly from this Colab session, run the following commands:

1. Expose the server using `localtunnel`:
```bash
# Install localtunnel
!npm install -g localtunnel

# Run Streamlit in the background and pipe output to localtunnel
!streamlit run app.py & npx localtunnel --port 8501
```

2. Copy the IP address printed by the following command (you'll need it as the password for localtunnel):
```python
!curl ipv4.icanhazip.com
```

3. Click the localtunnel URL (e.g. `https://xxxx.localtunnel.me`), paste the IP address, and click submit. Your interactive dashboard will open!